# Train FRM for real — does a learned module beat a geometric rule?

Everything so far has been proxies. At n=95 the picture is:

| | above chance | share of teacher signal |
|---|---|---|
| attn x grad **[sees the question]** | 18.0 pts | 100% |
| token norm (best question-free) | 8.8 pts | 49% |
| **gaze blob (pure geometry)** | **6.3 pts** | **35%** |
| gaze-sim = FRM with identity projections | 1.3 pts | 7% *(not above random)* |

So FRM has to live in the band **22.1% -> ~24.6%**, and the untrained version of its own
architecture is indistinguishable from random. The one defence left is that **identity projections
are a weak floor** — the whole point of learning `W_q, W_k` is to encode associations that raw
cosine similarity cannot see ("you are looking at a product, so look for a price tag").

This notebook removes that defence one way or the other.

## What it does

* Trains the FRM head from spec section 3: `s = (W_k G)(W_q gaze) / sqrt(d_h)`, multi-head,
  fovea and sinks excluded from the candidate set.
* Distils from the **LOO ground truth** — the exact ablation labels, not an attention proxy.
* **5-fold cross-validation**, so every one of the ~100 examples gets a held-out prediction and the
  reported number is never fitted on itself.
* Scores it on the same referee (precision@10, fovea excluded) against the geometric baseline it
  must beat, plus a **linear probe over all question-free features** — the "combine everything"
  ceiling I have been claiming is the last open door.

## The decision rule, fixed in advance

* **FRM clears the gaze blob by >= 5 points on held-out folds** -> the learned projections found
  something identity could not. Build it, scale the labels.
* **FRM lands at 22-25%** -> it is matching a 3-parameter geometric rule. Spec kill criterion fires:
  drop Stage 2b, use eccentricity.
* **The linear probe beats FRM** -> the signal is in generic question-free features, not in the
  gaze-to-context association. Reframe the module.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, gc, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import wilcoxon

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import visual_selection as VS

N100  = "/content/drive/MyDrive/wearvqa_n100.pt"
EMB   = "/content/drive/MyDrive/wearvqa_n100_emb.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

assert os.path.exists(N100), (
    f"{N100} not found - run colab_n100_scaleup.ipynb first (it builds the LOO labels)")
data  = torch.load(N100, weights_only=False)
data  = [d for d in data if "gp" in d and "drops" in d]
sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()

L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)
print(f"{N} examples | L_v={L_v} ({G}x{G}) | sinks {int(sinks.sum())}")

## 2. Token embeddings

The n=100 run cached *derived* features (token norm, distinctiveness, gaze similarity) but not the
raw `[L_v, d]` embeddings, and FRM needs those to learn `W_q, W_k`. Image-token embeddings at the
input to decoder layer 0 are the projected vision features, so they do not depend on the prompt —
one cheap forward per image.

In [ ]:
if os.path.exists(EMB):
    embs = torch.load(EMB, weights_only=False)
    print(f"loaded {len(embs)} cached embeddings")
else:
    model, processor, device = S._load_smolvlm(MODEL_ID)
    tok = processor.tokenizer

    def find_decoder_layers(model, n=24):
        hits = [(nm, m) for nm, m in model.named_modules()
                if isinstance(m, torch.nn.ModuleList) and len(m) == n]
        for nm, m in hits:
            if any(t in nm for t in ("text", "language", "llm")):
                return m
        return hits[0][1]
    dec = find_decoder_layers(model)

    embs, t0 = [], time.time()
    for i, d in enumerate(data):
        img = S.load_image(d["img_path"])
        msgs = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": "Describe the image."}]}]
        inp = processor(text=processor.apply_chat_template(msgs, add_generation_prompt=True),
                        images=[img], return_tensors="pt").to(device)
        ids = inp["input_ids"][0].cpu()
        cols = torch.nonzero(ids == S._find_image_token_id(model, processor)).squeeze(-1)
        store = {}
        h = dec[0].register_forward_pre_hook(lambda _m, a: store.__setitem__("h", a[0]))
        try:
            with torch.no_grad():
                model(**inp)
        finally:
            h.remove()
        embs.append(store["h"][0].detach().float().cpu()[cols])
        del store, inp
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{N}  ({(time.time()-t0)/60:.1f} min)")
    torch.save(embs, EMB)
    del model; gc.collect(); torch.cuda.empty_cache()

d_model = embs[0].shape[-1]
print(f"embeddings: {len(embs)} x {tuple(embs[0].shape)}  (d_model={d_model})")

## 3. FRM, the teacher distribution, and the candidate set

In [ ]:
def fovea_mask(gp):
    r0, c0 = divmod(gp, G)
    m = torch.zeros(L_v, dtype=torch.bool)
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            r, c = r0 + dr, c0 + dc
            if 0 <= r < G and 0 <= c < G:
                m[r * G + c] = True
    return m

def cand_for(d):                      # spec 3.1: sinks AND the fovea leave the candidate set
    return VS.candidate_mask(L_v, exclude=[sinks[:L_v], fovea_mask(d["gp"])])

TEMP = 1.0
def teacher_dist(drops, cand):
    """Softmax over candidates, per-example scale. Dense, and handles negative drops
    (ablation that HELPED) naturally, unlike relu+renormalise."""
    x = drops[cand].float()
    return F.softmax(x / (x.std().clamp_min(1e-6) * TEMP), dim=0)

def blob(gp, sc=2.69, sr=1.72, oc=0.95, orr=-0.20):
    r0, c0 = divmod(gp, G); r0, c0 = r0 + orr, c0 + oc
    return torch.tensor([-(((i//G - r0)/sr)**2 + ((i%G - c0)/sc)**2) for i in range(L_v)])

def isotropic(gp):
    r0, c0 = divmod(gp, G)
    return torch.tensor([-math.hypot(i//G - r0, i % G - c0) for i in range(L_v)])


class FRM(nn.Module):
    """Spec section 3:  s = (W_k G)(W_q gaze) / sqrt(d_h), multi-head, averaged."""
    def __init__(self, d, d_h=64, heads=1, use_prior=False):
        super().__init__()
        self.Wq = nn.Linear(d, d_h, bias=False)
        self.Wk = nn.Linear(d, d_h, bias=False)
        self.h  = heads
        self.dh = d_h // heads
        self.prior = nn.Parameter(torch.tensor(0.0)) if use_prior else None

    def forward(self, Eg, E, prior=None):
        q = self.Wq(Eg).view(self.h, self.dh)                 # [H, dh]
        k = self.Wk(E).view(-1, self.h, self.dh)              # [T, H, dh]
        s = (k * q.unsqueeze(0)).sum(-1) / math.sqrt(self.dh) # [T, H]
        s = s.mean(-1)                                        # [T]
        if self.prior is not None and prior is not None:
            s = s + self.prior * prior
        return s


class Probe(nn.Module):
    """Linear probe over every question-free feature -- the 'combine everything' ceiling."""
    FEATS = ("tok_norm", "distinct", "gaze_sim")
    def __init__(self, n_extra=2):
        super().__init__()
        self.w = nn.Linear(len(self.FEATS) + n_extra, 1)

    @staticmethod
    def features(d, cand):
        def z(v):
            v = v.float(); return (v - v.mean()) / v.std().clamp_min(1e-6)
        cols = [z(d[f])[cand] for f in Probe.FEATS if f in d]
        cols += [z(blob(d["gp"]))[cand], z(isotropic(d["gp"]))[cand]]
        return torch.stack(cols, dim=-1)

    def forward(self, X):
        return self.w(X).squeeze(-1)

print("ok")

## 4. 5-fold cross-validation

Every example gets a held-out prediction, so nothing is scored on data it was fitted to. Folds are
stratified by question type.

In [ ]:
K_EVAL, EPOCHS, LR, WD, PATIENCE = 10, 400, 3e-3, 1e-2, 60
D_H, HEADS = 64, 1
torch.manual_seed(0)

# stratified folds
by_type = defaultdict(list)
for i, d in enumerate(data):
    by_type[d["type"]].append(i)
folds = [[] for _ in range(5)]
for t, idxs in by_type.items():
    for j, i in enumerate(idxs):
        folds[j % 5].append(i)
print("fold sizes:", [len(f) for f in folds])

PRE = []
for i, d in enumerate(data):
    c = cand_for(d)
    PRE.append(dict(cand=c, E=embs[i][c], Eg=embs[i][d["gp"]],
                    p=teacher_dist(d["drops"], c), prior=blob(d["gp"])[c],
                    X=Probe.features(d, c), drops=d["drops"][c],
                    gt=set(torch.topk(d["drops"][c], K_EVAL).indices.tolist())))

def prec(scores, gt):
    return len(set(torch.topk(scores, K_EVAL).indices.tolist()) & gt) / K_EVAL

def train_eval(make_model, fwd, tr, te, epochs=EPOCHS):
    m = make_model()
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    best, best_state, bad = 1e9, None, 0
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss = sum(F.kl_div(F.log_softmax(fwd(m, PRE[i]), 0), PRE[i]["p"],
                            reduction="sum") for i in tr) / len(tr)
        loss.backward(); opt.step()
        with torch.no_grad():                       # early stop on TRAIN loss (no val leak)
            if loss.item() < best - 1e-5:
                best, best_state, bad = loss.item(), {k: v.clone() for k, v in m.state_dict().items()}, 0
            else:
                bad += 1
                if bad > PATIENCE:
                    break
    m.load_state_dict(best_state); m.eval()
    with torch.no_grad():
        tr_p = float(np.mean([prec(fwd(m, PRE[i]), PRE[i]["gt"]) for i in tr]))
        te_p = [prec(fwd(m, PRE[i]), PRE[i]["gt"]) for i in te]
    return tr_p, te_p

ARMS = {
    "FRM (gaze query)":       (lambda: FRM(d_model, D_H, HEADS),
                               lambda m, r: m(r["Eg"], r["E"])),
    "FRM + geometry prior":   (lambda: FRM(d_model, D_H, HEADS, use_prior=True),
                               lambda m, r: m(r["Eg"], r["E"], r["prior"])),
    "linear probe (all feats)":(lambda: Probe(),
                               lambda m, r: m(r["X"])),
}

held = defaultdict(lambda: [None] * N)
traingap = defaultdict(list)
for fi, te in enumerate(folds):
    tr = [i for i in range(N) if i not in set(te)]
    for name, (mk, fwd) in ARMS.items():
        tp, tep = train_eval(mk, fwd, tr, te)
        traingap[name].append(tp)
        for j, i in enumerate(te):
            held[name][i] = tep[j]
    print(f"  fold {fi+1}/5 done")

for name in ARMS:
    print(f"{name:<26} train {np.mean(traingap[name]):.1%}  "
          f"held-out {np.mean(held[name]):.1%}  "
          f"(gap {np.mean(traingap[name]) - np.mean(held[name]):+.1%})")

## 5. Held-out comparison against the baselines it must beat

In [ ]:
g = torch.Generator().manual_seed(0)
BASE = {
    "gaze blob (geometry)": lambda i, r: blob(data[i]["gp"])[r["cand"]],
    "isotropic gaze prox":  lambda i, r: isotropic(data[i]["gp"])[r["cand"]],
    "token norm (image)":   lambda i, r: data[i]["tok_norm"][r["cand"]],
    "gaze-sim (untrained)": lambda i, r: data[i]["gaze_sim"][r["cand"]],
    "CTRL random":          lambda i, r: torch.rand(int(r["cand"].sum()), generator=g),
}
rows = {n: [prec(f(i, PRE[i]), PRE[i]["gt"]) for i in range(N)] for n, f in BASE.items()}
for n in ARMS:
    rows[n] = list(held[n])

chance = float(np.mean([K_EVAL / int(PRE[i]["cand"].sum()) for i in range(N)]))
order = sorted(rows, key=lambda n: -np.mean(rows[n]))
ref = "gaze blob (geometry)"
print(f"precision@{K_EVAL}, fovea excluded, HELD-OUT   chance {chance:.1%}   n={N}\n")
print(f"{'predictor':<28}{'prec':>7}{'95% CI':>16}{'vs blob':>9}{'vs rand':>9}")
print("-" * 70)
for n in order:
    a = np.array(rows[n]); se = a.std(ddof=1) / math.sqrt(len(a))
    def pw(b):
        b = np.array(b)
        try:
            return "  --" if np.allclose(a, b) else f"{wilcoxon(a, b)[1]:.3f}"
        except Exception:
            return " n/a"
    star = "  <-- FRM" if n in ARMS else ""
    print(f"{n:<28}{a.mean():>7.1%}  [{a.mean()-1.96*se:>5.1%},{a.mean()+1.96*se:>6.1%}]"
          f"{pw(rows[ref]):>9}{pw(rows['CTRL random']):>9}{star}")

gain = np.mean(rows["FRM (gaze query)"]) - np.mean(rows[ref])
print(f"\nFRM minus geometry: {gain:+.1%}  "
      f"(decision rule: >= +5 pts -> build; 0-3 pts -> drop Stage 2b)")

In [ ]:
# capacity sweep -- is the ceiling the model or the data? (spec Exp 7, mini version)
print(f"{'d_h':>5}{'heads':>7}{'train':>9}{'held-out':>11}{'gap':>8}")
print("-" * 40)
for dh, hh in [(16, 1), (64, 1), (64, 4), (256, 4)]:
    tr_all, te_all = [], [None] * N
    for te in folds:
        tr = [i for i in range(N) if i not in set(te)]
        tp, tep = train_eval(lambda: FRM(d_model, dh, hh),
                             lambda m, r: m(r["Eg"], r["E"]), tr, te)
        tr_all.append(tp)
        for j, i in enumerate(te):
            te_all[i] = tep[j]
    print(f"{dh:>5}{hh:>7}{np.mean(tr_all):>9.1%}{np.mean(te_all):>11.1%}"
          f"{np.mean(tr_all)-np.mean(te_all):>+8.1%}")

## 6. Verdict

Read the **held-out** column, and the `vs blob` p-value.

* **FRM >= blob + 5 pts, p < 0.05** -> learned `W_q, W_k` extract a gaze-to-context association that
  identity projections could not. The premise holds. Generate LOO labels at scale and continue to
  spec Exp 1.
* **FRM within 0-3 pts of the blob** -> a ~1M-parameter learned module matches a 3-parameter
  geometric rule. The spec's own kill criterion fires ("FRM ~ Eccentricity -> DROP FRM"). Use
  eccentricity for Stage 2b and redirect the effort.
* **Linear probe > FRM** -> what signal exists is in generic question-free features (token norm,
  distinctiveness), not in the gaze-to-context association. The honest module is "learned saliency",
  and gaze is not what makes it work.
* **Large train/held-out gap in the capacity sweep** -> the limit is 100 examples, not the
  architecture. Regenerate with a larger `N_PER_TYPE` before concluding anything (the dataset holds
  68-104 per type, so ~680 total is available).

Caveat kept in view: the teacher here is the LOO ground truth itself, so this is the most favourable
setting FRM will ever get — no proxy loss in between. A result that fails here will not be rescued by
a better teacher.